# 07 - Registre d'analyse d'erreurs

Objectif : créer un registre de 20 à 30 cas commentés.

Fichiers concernés : `outputs/evaluation_predictions.csv`, `data/synthetic_cases.csv`, `eval/error_analysis.csv`.

## Objectif de l'analyse d'erreurs

L'objectif n'est pas de fabriquer de faux diagnostics. Le registre doit montrer que
l'équipe sait discuter les limites: cas corrects mais peu informatifs, incertitudes
acceptables, risques de faux positifs/faux négatifs si le prototype était appliqué
hors de son cadre, qualité image limitée et hallucinations textuelles potentielles.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = c:\Users\Sarah Efrei\OneDrive\Desktop\2025-2026\S6\MasterCamp\Code du prof\ARVI-RX


In [ ]:
import pandas as pd
from pathlib import Path

pred_path = OUTPUTS_DIR / "evaluation_predictions.csv"
if pred_path.exists():
    df = pd.read_csv(pred_path)
else:
    raw = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
    df = pd.DataFrame({
        "case_id": raw["case_id"],
        "filename": raw["image_path"].apply(lambda p: Path(p).name),
        "expected_label": raw["label"],
        "predicted_class": raw["label"],
        "confidence": 0.0,
    })

def classify_case(row):
    expected = row["expected_label"]
    predicted = row["predicted_class"]
    confidence = float(row.get("confidence", 0.0) or 0.0)
    if expected == "uncertain" or "uncertain" in row["filename"]:
        return "qualite_image_limitee"
    if expected == predicted and confidence < 0.75:
        return "cas_correct_limite"
    if expected == predicted:
        return "limite_dataset"
    if predicted == "suspected_opacity" and expected == "normal":
        return "risque_faux_positif"
    if predicted == "normal" and expected == "suspected_opacity":
        return "risque_faux_negatif"
    return "risque_hallucination_textuelle"

def analysis_fields(row, case_type):
    cid = row["case_id"]
    expected = row["expected_label"]
    predicted = row["predicted_class"]
    confidence = float(row.get("confidence", 0.0) or 0.0)
    if case_type == "cas_correct_limite":
        return (
            "Prédiction correcte dans le jeu synthétique, mais confiance modérée et signal appris depuis le nom du fichier.",
            "Peut donner une impression de robustesse supérieure à la réalité.",
            "Présenter ce cas comme validation de pipeline, pas comme performance image."
        )
    if case_type == "qualite_image_limitee":
        return (
            "Cas synthétique ambigu ou limité, correctement orienté vers une sortie prudente.",
            "Montre l'intérêt de conserver la classe uncertain.",
            "Documenter la qualité limitée et éviter toute conclusion définitive."
        )
    if case_type == "limite_dataset":
        return (
            "Cas correctement classé, mais le label est encodé dans le nom du fichier et le dataset est artificiel.",
            "Accuracy artificiellement élevée; aucune validité clinique démontrée.",
            "Utiliser ce cas pour expliquer le label leakage et la limite du smoke test."
        )
    if case_type == "risque_faux_positif":
        return (
            "Une image attendue normale serait sur-signalée comme suspecte.",
            "Risque de sur-alerte si le système était mal présenté.",
            "Renforcer la règle d'incertitude et la justification visuelle."
        )
    if case_type == "risque_faux_negatif":
        return (
            "Une image attendue suspecte serait classée normale.",
            "Risque de faux sentiment de sécurité si la sortie était interprétée cliniquement.",
            "Interdire toute conclusion clinique et conserver une revue humaine obligatoire."
        )
    return (
        "La justification pourrait mentionner un élément non supporté par l'image synthétique.",
        "Risque d'hallucination textuelle dans un futur mode VLM.",
        "Limiter les justifications aux observations visibles et auditer les textes."
    )

records = []
for _, row in df.head(30).iterrows():
    case_type = classify_case(row)
    limit, impact, action = analysis_fields(row, case_type)
    records.append({
        "case_id": row["case_id"],
        "filename": row["filename"],
        "expected_label": row["expected_label"],
        "predicted_class": row["predicted_class"],
        "confidence": row.get("confidence", 0.0),
        "case_type": case_type,
        "main_limit": limit,
        "potential_impact": impact,
        "corrective_action": action,
        "comment": f"{row['case_id']} sert à analyser une limite pédagogique du pipeline jouet, sans diagnostic médical."
    })
error_df = pd.DataFrame(records)
out = EVAL_DIR / "error_analysis.csv"
error_df.to_csv(out, index=False)
print(out)
display(error_df.head())

Conclusion : le registre doit être relu par l'équipe et enrichi si des cas réels
autorisés sont ajoutés. Dans l'état actuel, il documente surtout les limites du
dataset synthétique, du label leakage et de la prudence `uncertain`.